# DistilBERT — twitter+airlinequality (re-run)

This notebook re-trains **only the missing `distilbert/twitter+airlinequality` model**.
The other 55 experiments were already completed; during the original run this model was
lost because its zip-download step pointed to the wrong path (`03_transformers.ipynb`
cell 14 bug).

**Expected test metrics** (from `experiment_log_transformers.csv`):
- Accuracy ≈ 0.8295
- Macro-F1 ≈ 0.7303

Because of hardware-level randomness the exact numbers may differ slightly, but should be
within about ±0.01.

## Usage
1. Open a GPU runtime in Colab (T4 is enough, ~20-30 min).
2. The files `twitter+airlinequality_{train,val,test}.csv` must be under `MyDrive/PATH/TO/data/splits/` in your Drive.
3. Run all cells.
4. The trained model and metric CSV are **written directly to Drive** (`MyDrive/PATH/TO/models/distilbert/twitter+airlinequality/` and `.../results/`).
5. Optional: the last cell also produces a zip for local download (skip it if you have Drive access).

## 1. Setup

In [ ]:
!pip install -q transformers

from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE = "/content/drive/MyDrive/PATH/TO"  # TODO: set to your Google Drive project path
SPLITS_DIR = f"{DRIVE_BASE}/data/splits"
RESULTS_DIR = f"{DRIVE_BASE}/results"
MODELS_DIR = f"{DRIVE_BASE}/models"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

SOURCE_CONFIG = "twitter+airlinequality"
MODEL_KEY = "distilbert"
MODEL_NAME = "distilbert-base-uncased"

for split in ("train", "val", "test"):
    path = os.path.join(SPLITS_DIR, f"{SOURCE_CONFIG}_{split}.csv")
    assert os.path.exists(path), f"MISSING: {path}"
    print(f"OK  {path}")

In [ ]:
import csv
import random
from datetime import datetime

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup

# ====== Config (identical to 03_transformers.ipynb) ======
SEED = 42
LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_NAMES = ["negative", "neutral", "positive"]

# twitter+airlinequality mix -> use the review max_length (same as the original rule)
MAX_LENGTH = 512
EPOCHS = 4
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Data + Model Helpers

In [ ]:
def load_split(source_config, split):
    path = os.path.join(SPLITS_DIR, f"{source_config}_{split}.csv")
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            texts.append(row["text"])
            labels.append(LABEL_MAP[row["label"]])
    return texts, labels


class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += outputs.loss.item() * len(labels)
        correct += (outputs.logits.argmax(1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, correct / total


def evaluate_model(model, loader):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item() * len(labels)
            all_preds.extend(outputs.logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(all_labels), np.array(all_preds), np.array(all_labels)


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro"),
        "macro_recall": recall_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }


def compute_per_class_metrics(y_true, y_pred):
    result = {}
    for avg in ["macro", "weighted", None]:
        p = precision_score(y_true, y_pred, average=avg, zero_division=0)
        r = recall_score(y_true, y_pred, average=avg, zero_division=0)
        f = f1_score(y_true, y_pred, average=avg, zero_division=0)
        if avg is None:
            for i, name in enumerate(LABEL_NAMES):
                result[f"{name}_precision"] = p[i]
                result[f"{name}_recall"] = r[i]
                result[f"{name}_f1"] = f[i]
        else:
            result[f"{avg}_precision"] = p
            result[f"{avg}_recall"] = r
            result[f"{avg}_f1"] = f
    return result

print("Helpers ready.")

## 3. Train

In [ ]:
set_seed()

print(f"[TRAIN] {MODEL_KEY} ({MODEL_NAME}) | {SOURCE_CONFIG}")
print(f"  max_length={MAX_LENGTH}, epochs={EPOCHS}, batch={BATCH_SIZE}")

train_texts, train_labels = load_split(SOURCE_CONFIG, "train")
val_texts,   val_labels   = load_split(SOURCE_CONFIG, "val")
test_texts,  test_labels  = load_split(SOURCE_CONFIG, "test")
print(f"  sizes: train={len(train_texts)} val={len(val_texts)} test={len(test_texts)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds = TextDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_ds   = TextDataset(val_texts,   val_labels,   tokenizer, MAX_LENGTH)
test_ds  = TextDataset(test_texts,  test_labels,  tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

best_val_f1 = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_preds, val_labels_arr = evaluate_model(model, val_loader)
    val_f1 = f1_score(val_labels_arr, val_preds, average="macro")
    print(f"  Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_f1={val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\nBest val Macro-F1: {best_val_f1:.4f}")

## 4. Test + Save

In [ ]:
model.load_state_dict(best_state)
model = model.to(device)

_, test_preds, test_labels_arr = evaluate_model(model, test_loader)

metrics = compute_metrics(test_labels_arr, test_preds)
per_class = compute_per_class_metrics(test_labels_arr, test_preds)
cm = confusion_matrix(test_labels_arr, test_preds)

print(f"TEST | Accuracy: {metrics['accuracy']:.4f} | Macro-F1: {metrics['macro_f1']:.4f}")
print(f"     | Beklenen ≈ accuracy 0.8295, macro_f1 0.7303 (±0.01)")
print(f"Confusion Matrix (rows=true, cols=pred — order: negative, neutral, positive):\n{cm}")

# Save model + tokenizer
save_dir = os.path.join(MODELS_DIR, MODEL_KEY, SOURCE_CONFIG)
os.makedirs(save_dir, exist_ok=True)
model.cpu().save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"\nModel saved to: {save_dir}")

# Single-row metric log (opsiyonel — projedeki master log'a manuel ekleyebilirsin)
row = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "model": MODEL_KEY,
    "source_config": SOURCE_CONFIG,
    **metrics, **per_class,
}
log_path = os.path.join(RESULTS_DIR, "distilbert_twitter_airlinequality_rerun.csv")
with open(log_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(row.keys()))
    writer.writeheader()
    writer.writerow(row)
print(f"Metric row saved to: {log_path}")

## 5. (Optional) Zip + Local Download

The model is already on Drive (`MyDrive/PATH/TO/models/distilbert/twitter+airlinequality/`). Skip this cell if you sync Drive locally. Otherwise download the zip and extract it under `models/distilbert/twitter+airlinequality/` in the project.

In [ ]:
import subprocess

src = os.path.join(MODELS_DIR, MODEL_KEY, SOURCE_CONFIG)
zip_path = f"{DRIVE_BASE}/model_distilbert_twitter_airlinequality.zip"

# No -j flag -- keep the original folder hierarchy
subprocess.run(["zip", "-r", zip_path, src], check=True)
print(f"\nZip ready (persisted on Drive): {zip_path}")
print("Download it from the Drive UI, or pull it to the browser with files.download() below.")

# Direct download to the browser (optional):
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print(f"(google.colab.files not available: {e}) -- download manually from the Drive UI.")